<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Leveraging GenAI as a Data Scientist

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/genai/02-leveraging-genai-data-science/02_leveraging_genai_as_a_data_scientist.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

---

In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/genai/02-leveraging-genai-data-science"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


### About
An introduction to how Generative AI can help data scientists work more efficiently.  This includes producing code, and parsing unstructured text data.  This module includes practical exercises and will emphasise the importance of sense checking the output of Generative AI.

### Learning Objectives
- Understand that GenAI can be used to write and document code more efficiently, and perform certain data analysis tasks.
- Gain practical experience in both of these applications.
- Appreciate that GenAI models are not perfectly accurate, and learn the importance of checking their output.

### Notebook Guide
- Exercise: Writing code with GPT
- Exercise: Writing documentation with GPT
- Using the Gemini code interface
- Exercise: Cleaning coffee price data with Gemini
- Stretch exercise: Analysing news data with Gemini

In [ ]:
%pip install -qqq requests beautifulsoup4 google-generativeai


# Using the Gemini code interface

Google Colab notebooks, like this one, have built-in support for sending prompts to Gemini. Let's try it out!

First, visit https://aistudio.google.com/app/apikey to create an API key. This is a password or key that lets you access Gemini.

Keep this key private and don't share it with anyone else. Enter it into the cell below:

In [ ]:
# Enter your API key from https://aistudio.google.com/app/apikey below
GOOGLE_API_KEY = "YOUR_KEY_HERE"

Next, try sending a prompt to Gemini. If you like, you can modify the cell below to change the prompt to something of your own choice.

Run the install cell near the top of the notebook if packages are missing.


In [ ]:
import google.generativeai as genai

prompt_text = "Give a very simple definition of what a Python library is"

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
for x in genai.list_models():
  print(x.name)

In [ ]:
model = genai.GenerativeModel("models/gemini-2.5-flash")

chat = model.start_chat(history=[])

response = chat.send_message(prompt_text)
print(response.text)

# Exercise: Cleaning coffee price data with Gemini

In this next exercise, we will use Gemini's code interface to clean up some coffee commodity price data.

The data consists of 100 data points, giving the price/kg of coffee on different dates.

First of all, we need to load our coffee data into this notebook. You will find instructions on how to do this in the slide deck.

Next, let's inspect a snapshot of the coffee data by running the cell below.

In [ ]:
import pandas as pd

# Load the data we generated in the previous step
coffee_prices = pd.read_csv('../data/coffee_prices.csv')
coffee_prices.head()

In [ ]:
# Generating a mock coffee_prices.csv for the exercise
import pandas as pd

data = {
    'Date': ['01-01-2023', '2nd of Jan 23', '2023/01/03', 'Jan 4 2023', '05.01.23'],
    'Price': ['$12.50', '13 dollars', '$12.80!', 'approx 14', '13.20 usd']
}

pd.DataFrame(data).to_csv('coffee_prices.csv', index=False)
print('coffee_prices.csv created successfully.')

Discuss with a partner:
* What do you notice about the data quality?
* Why would these quality issues be difficult to fix using conventional (i.e. non-AI) Python techniques?

Below, you will find some incomplete code.

The code takes the second row of the coffee data, and retrieves the date and price.

There are blank spaces for you to do the following:

* Use Gemini to standardise the messy original date into the format 'dd-mm-yyyy'

* Use Gemini to standardise the messy original price into the format '$XX.XX'

Finally, the code prints the cleaned up date and price data.

There are several different ways of solving this. Enjoy experimenting!

In [ ]:
import google.generativeai as genai
import pandas as pd

try:
    coffee_prices = pd.read_csv("../data/coffee_prices.csv")
    chat = model.start_chat(history=[])

    messy_date = coffee_prices.loc[1, "Date"]
    messy_price = coffee_prices.loc[1, "Price"]

    clean_date = chat.send_message(f"Convert to dd-mm-yyyy: {messy_date}").text
    clean_price = chat.send_message(f"Convert to $XX.XX: {messy_price}").text

    print(f"Original: {messy_date}, {messy_price}")
    print(f"Cleaned: {clean_date}, {clean_price}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
chat = model.start_chat(history=[])

def parse_date(d):
  return chat.send_message(f"Convert to dd-mm-yyyy: {d}").text

def parse_price(p):
  return chat.send_message(f"Convert to $XX.XX: {p}").text

In [ ]:
coffee_prices = pd.read_csv("../data/coffee_prices.csv")
coffee_prices.loc[:5, "Date"] = coffee_prices.loc[:5, "Date"].apply(parse_date)
coffee_prices.loc[:5, "Price"] = coffee_prices.loc[:5, "Price"].apply(parse_price)
coffee_prices.head()

Finally, **without** implementing the solution, discuss with a partner:

* How you might modify this code further to clean the entire dataset, instead of just one row
* How you might check whether Gemini had cleaned the data correctly

In [ ]:
import google.generativeai as genai

# Connect to Gemini
# model = genai.GenerativeModel('gemini-pro')
# Tell Gemini what your API key is
# genai.configure(api_key=GOOGLE_API_KEY)
# Begin a new chat with Gemini
chat = model.start_chat(history=[])

# Retrieve the date and coffee price in the second row (which has index '1')
messy_date = coffee_prices.loc[1, 'Date']
messy_price = coffee_prices.loc[1, 'Price']

# Use Gemini to standardise the messy original date into the format 'dd-mm-yyyy'
date_prompt = 'Reformat the following date into the format dd-mm-yyyy:'+messy_date
clean_date = chat.send_message(date_prompt).text

# Use Gemini to standardise the messy original price into the format '$XX.XX'
price_prompt = 'Reformat the following price into a standard numerical format. Do not give me any extra text or ask me any questions:'+messy_price
clean_price= chat.send_message(price_prompt).text

# Print the original date and price
print(f'Original date: {messy_date}')
print(f'Original price: {messy_price}')

# Print the cleaned up date and price
print(f'Cleaned up date: {clean_date}')
print(f'Cleaned up price: {clean_price}')

# Stretch exercise: Analysing news data with Gemini

In this exercise, you will use Gemini to analyse the text from online news articles.

Start by loading the file `cnn_articles.csv` into Colab.

This dataset consists of 4000 CNN articles pubished in 2023.

The source of this data is: https://www.kaggle.com/datasets/pedroaribe/4000-cnn-articles-as-of-1062023

Run the cell below to read the data and display a snapshot of the first 5 rows:

In [ ]:
import pandas as pd
# Creating a dummy cnn_articles.csv for exercise flow if not exists
try:
    news = pd.read_csv('../data/cnn_articles.csv')
except FileNotFoundError:
    data = {
        'Body': [
            'The stock market saw a significant rise today as tech giants reported earnings.',
            'A new species of frog was discovered in the Amazon rainforest last week.',
            'The election results are coming in from across the country.',
            'New travel restrictions have been announced for international flights.'
        ],
        'Theme': ['Business', 'Science', 'Politics', 'Travel']
    }
    news = pd.DataFrame(data)
    news.to_csv('cnn_articles.csv', index=False)
print('News data loaded successfully.')
news.head()

Now, use Gemini's coding interface to sort the **first** news article in the dataset into one of the following categories, based on the article **body text**:

```US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other```

You can do this by filling in the gaps in the code below.

How does Gemini's classification compare to the real category, which is displayed in the `Theme` column of the dataset?

What happens if you run the finished code repeatedly? Does the classification stay the same or change? What are the implications of this for accuracy and reliability?

Experiment with asking Gemini to classify different articles instead of just the first.

In [ ]:
import google.generativeai as genai
import pandas as pd

try:
    news = pd.read_csv("../data/cnn_articles.csv")
    article_text = news.loc[0, "Body"]
    prompt = f"Categorise this news article into one of the following categories: US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other. Output only the category name.\n\nArticle text: {article_text}"
    news_classification = model.generate_content(prompt).text

    print(f"Article: {article_text[:200]}...")
    print(f"Classification: {news_classification}")
except Exception as e:
    print(f"Error: {e}")

Can you modify your code to turn it into a function, which classifies the _Nth_ article in the dataset instead of just the first one?


In [ ]:
def classify_nth_article(n, df):
    """Classifies the Nth article in the dataframe using Gemini."""
    article_text = df.loc[n, "Body"]
    actual_theme = df.loc[n, "Theme"]

    model = genai.GenerativeModel("gemini-1.5-flash-latest")
    prompt = f"Categorise this news article into one of the following categories: US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other. Output only the category name.\n\nArticle text: {article_text}"

    gemini_category = model.generate_content(prompt).text

    print(f"Article Index: {n}")
    print(f"Gemini Category: {gemini_category.strip()}")
    print(f"Actual Theme: {actual_theme}")
    print("-" * 30)
    return gemini_category

Can you use this function to classify the first 10 articles in the dataset?

With a partner, discuss how you might check the accuracy of Gemini's classifications.

In [ ]:
# Classify the first 4 articles (indices 0 to 3)
for i in range(4):
    classify_nth_article(i, news)

Use Gemini's coding interface to sort the first news article in the dataset into one of the following categories, based on the article text:

US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other

In [ ]:
import google.generativeai as genai

# Connect to Gemini
model = genai.GenerativeModel('gemini-pro')
# Tell Gemini what your API key is
genai.configure(api_key=GOOGLE_API_KEY)
# Begin a new chat with Gemini
chat = model.start_chat(history=[])

# Retrieve the text of the first article.
article_text = news.loc[0, 'Body']
# Also retrieve the actual classification of the article, from the 'theme' column of the dataset
actual_news_classification = news.loc[0, 'Theme']

# Use Gemini to categorise the article
prompt = 'Categorise this news headline into one of the following categories: US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other. The headline to classify is:'+article_text
news_classification = chat.send_message(prompt).text

# Print the article text, its classification from Gemini, and its actual classification from the
# 'theme' column of the dataset
print(f'Article: {article_text}')
print(f'Gemini classification: {news_classification}')
print(f'Actual classification: {actual_news_classification}')

Can you modify your code to turn it into a function, which classifies the _Nth_ article in the dataset instead of just the first one?


In [ ]:
import google.generativeai as genai

def classify_nth_article(article_number,news):

  # Connect to Gemini
  model = genai.GenerativeModel('gemini-pro')
  # Tell Gemini what your API key is
  genai.configure(api_key=GOOGLE_API_KEY)
  # Begin a new chat with Gemini
  chat = model.start_chat(history=[])

  # Retrieve the text of the first article.
  article_text = news.loc[article_number-1, 'Body']
  # Also retrieve the actual classification of the article, from the 'theme' column of the dataset
  actual_news_classification = news.loc[article_number-1, 'Theme']

  # Use Gemini to categorise the article
  prompt = 'Categorise this news headline into one of the following categories: US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather, Other. The headline to classify is:'+article_text
  news_classification = chat.send_message(prompt).text

  # Print the article text, its classification from Gemini, and its actual classification from the
  # 'theme' column of the dataset
  print(f'Article: {article_text}')
  print(f'Gemini classification: {news_classification}')
  print(f'Actual classification: {actual_news_classification}')

  return news_classification

Can you use this function to classify the first 4 articles in the dataset?

With a partner, discuss how you might check the accuracy of Gemini's classifications.

In [ ]:
for article_number in range(1, 5):
  classify_nth_article(article_number, news)